# Jon2 — Private Safari Bundle Export

This notebook only exports the **existing original Jon2 checkpoint** into a browser-readable Float32 bundle. It does not train, modify, or upload model weights. Keep the output ZIP private.

**Before running:** open this notebook in your Kaggle account that has access to `jon2kabeel/jon2-private-model-checkpoint`. Enable Internet if Kaggle requires it for `kagglehub` access. Run cells from top to bottom.


In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, zipfile
import torch
import kagglehub

DATASET_SLUG = 'jon2kabeel/jon2-private-model-checkpoint'
EXPECTED_SIZE = 3_285_607
EXPECTED_SHA256 = 'acbae28e4acc4dc060194df466f65a72410792f5b0fec7ff9bdd939244b9c43f'
OUT_DIR = Path('/kaggle/working/jon2-safari-private')
OUT_DIR.mkdir(parents=True, exist_ok=True)
print('PyTorch:', torch.__version__)
print('Downloading the private dataset through your Kaggle account…')
dataset_dir = Path(kagglehub.dataset_download(DATASET_SLUG))
print('Dataset mounted at:', dataset_dir)


In [ ]:
def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

candidates = [p for p in dataset_dir.rglob('*') if p.is_file() and p.suffix.lower() in {'.pt', '.pth'}]
verified = []
for path in candidates:
    size = path.stat().st_size
    digest = sha256_file(path)
    print(path.name, size, digest)
    if size == EXPECTED_SIZE and digest == EXPECTED_SHA256:
        verified.append(path)
if len(verified) != 1:
    raise RuntimeError(f'Expected exactly one original Jon2 checkpoint matching the pinned fingerprint; found {len(verified)}. No export performed.')
checkpoint_path = verified[0]
print('Verified original checkpoint:', checkpoint_path.name)


In [ ]:
checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=True)
if not isinstance(checkpoint, dict) or not {'model_state', 'tokenizer', 'config'}.issubset(checkpoint):
    raise RuntimeError('Unsupported native Jon2 checkpoint structure.')

tok = checkpoint['tokenizer']
if tok.get('format') != 'jon2-byte-bpe-v1' or tok.get('base_vocab_size') != 256:
    raise RuntimeError('Unsupported original Jon2 tokenizer format.')
merges = tok.get('merges')
if not isinstance(merges, list):
    raise RuntimeError('Invalid tokenizer merge list.')
for token_id, pair in enumerate(merges, start=256):
    if not isinstance(pair, list) or len(pair) != 2 or any(type(x) is not int or x < 0 or x >= token_id for x in pair):
        raise RuntimeError(f'Invalid tokenizer merge at token {token_id}.')

from torch import nn
class Jon2Transformer(nn.Module):
    def __init__(self, vocab_size, width=128, heads=4, layers=4, context=128, dropout=0.1):
        super().__init__()
        if width % heads: raise ValueError('width must be divisible by heads')
        self.context = context
        self.token_embedding = nn.Embedding(vocab_size, width)
        self.position_embedding = nn.Embedding(context, width)
        block = nn.TransformerEncoderLayer(d_model=width, nhead=heads, dim_feedforward=4*width,
            dropout=dropout, batch_first=True, norm_first=True, activation='gelu')
        self.blocks = nn.TransformerEncoder(block, num_layers=layers)
        self.final_norm = nn.LayerNorm(width)
        self.output = nn.Linear(width, vocab_size, bias=False)
        self.output.weight = self.token_embedding.weight
    def forward(self, token_ids):
        _, length = token_ids.shape
        if length > self.context:
            token_ids = token_ids[:, -self.context:]
            length = self.context
        positions = torch.arange(length, device=token_ids.device)
        hidden = self.token_embedding(token_ids) + self.position_embedding(positions)[None, :, :]
        mask = torch.triu(torch.ones(length, length, device=token_ids.device, dtype=torch.bool), diagonal=1)
        hidden = self.blocks(hidden, mask=mask)
        return self.output(self.final_norm(hidden))

config = dict(checkpoint['config'])
model = Jon2Transformer(vocab_size=256 + len(merges), **config)
model.load_state_dict(checkpoint['model_state'], strict=True)
model.eval()
state = model.state_dict()
if not torch.equal(state['output.weight'], state['token_embedding.weight']):
    raise RuntimeError('Tied Jon2 output embedding check failed.')
print('Original architecture and all checkpoint tensors loaded strictly.')


In [ ]:
def ordered_tensor_names(cfg):
    names = ['token_embedding.weight', 'position_embedding.weight']
    for layer in range(int(cfg.get('layers', 4))):
        prefix = f'blocks.layers.{layer}.'
        names.extend(prefix + suffix for suffix in (
            'self_attn.in_proj_weight', 'self_attn.in_proj_bias',
            'self_attn.out_proj.weight', 'self_attn.out_proj.bias',
            'linear1.weight', 'linear1.bias', 'linear2.weight', 'linear2.bias',
            'norm1.weight', 'norm1.bias', 'norm2.weight', 'norm2.bias'))
    names.extend(['final_norm.weight', 'final_norm.bias'])
    return names

binary = bytearray()
tensor_manifest = {}
for name in ordered_tensor_names(config):
    if name not in state:
        raise RuntimeError('Missing original Jon2 tensor: ' + name)
    tensor = state[name].detach().cpu().contiguous().to(torch.float32)
    raw = tensor.numpy().astype('<f4', copy=False).tobytes(order='C')
    tensor_manifest[name] = {'offset': len(binary), 'elements': tensor.numel(), 'shape': list(tensor.shape)}
    binary.extend(raw)

weights_path = OUT_DIR / 'weights.bin'
weights_path.write_bytes(binary)
weights_sha = hashlib.sha256(binary).hexdigest()
manifest = {
    'format': 'jon2-safari-f32-v1',
    'checkpoint': {'size_bytes': EXPECTED_SIZE, 'sha256': EXPECTED_SHA256},
    'config': {'vocab_size': 256 + len(merges), **config},
    'tokenizer': tok,
    'byte_length': len(binary),
    'weights_sha256': weights_sha,
    'tensor_order': list(tensor_manifest),
    'tensors': tensor_manifest,
    'warning': 'PRIVATE MODEL DATA. Never commit or publish this manifest or weights file.'
}
(OUT_DIR / 'manifest.json').write_text(json.dumps(manifest, ensure_ascii=False, separators=(',', ':')), encoding='utf-8')

zip_path = Path('/kaggle/working/jon2-safari-private-bundle.zip')
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_STORED) as z:
    z.write(OUT_DIR / 'manifest.json', 'manifest.json')
    z.write(weights_path, 'weights.bin')
print('PRIVATE BUNDLE READY:', zip_path)
print('Checkpoint SHA-256 (pin):', EXPECTED_SHA256)
print('Weights SHA-256 (copy this separately):', weights_sha)
print('Bundle size:', zip_path.stat().st_size, 'bytes')


In [ ]:
from IPython.display import FileLink, display
display(FileLink('/kaggle/working/jon2-safari-private-bundle.zip'))
print('Download the ZIP to your iPhone Files app. Keep it private. Do not upload it to GitHub.')
